# VideoLLaMA3 Colab Test Notebook (Fixed Version)

This notebook demonstrates how to use VideoLLaMA3, a series of multimodal foundation models for image and video understanding, in Google Colab with GPU support.

## Models Available:
- **VideoLLaMA3-7B**: Based on Qwen2.5-7B (Full video understanding)
- **VideoLLaMA3-2B**: Based on Qwen2.5-1.5B (Lightweight video understanding)
- **VideoLLaMA3-7B-Image**: Based on Qwen2.5-7B (Image-focused)
- **VideoLLaMA3-2B-Image**: Based on Qwen2.5-1.5B (Lightweight image-focused)

## Features:
- Video question answering
- Video captioning and description
- Temporal understanding
- Visual reasoning
- Multi-image comparison

## Requirements:
- Python >= 3.10
- PyTorch >= 2.4.0
- CUDA Version >= 11.8
- GPU runtime enabled in Colab

## Troubleshooting Import Errors:
If you encounter `VideoInput` import errors, this version includes:
- Fallback mechanisms for video processing
- Alternative model loading strategies
- Text-only processing modes
- Comprehensive error handling

## 1. GPU Setup and Environment Check

In [ ]:
# Check GPU availability and specifications
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU detected! Please enable GPU runtime in Colab:")
    print("Runtime → Change runtime type → Hardware accelerator → GPU")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Installation and Dependencies

In [ ]:
# Install core dependencies with specific versions for compatibility
!pip install torch==2.4.0 torchvision==0.19.0 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.46.3 accelerate==1.0.1
!pip install flash-attn==2.7.3 --no-build-isolation

# Install video processing dependencies
!pip install decord ffmpeg-python imageio opencv-python-headless
!pip install pillow matplotlib ipywidgets

# Install additional utilities
!pip install gdown

print("✅ All dependencies installed successfully!")

# Check for potential import issues
try:
    import transformers.image_utils
    if not hasattr(transformers.image_utils, 'VideoInput'):
        print("⚠️  VideoInput not found - video processing may be limited")
        print("💡 Tip: Try VideoLLaMA3-2B-Image for image-only tasks")
except ImportError as e:
    print(f"⚠️  Import warning: {e}")
    print("💡 Tip: Using image models may work better")

## 3. Model Loading and Initialization

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
import torch

# Model configuration
MODEL_OPTIONS = {
    "VideoLLaMA3-7B": "DAMO-NLP-SG/VideoLLaMA3-7B",
    "VideoLLaMA3-2B": "DAMO-NLP-SG/VideoLLaMA3-2B",
    "VideoLLaMA3-7B-Image": "DAMO-NLP-SG/VideoLLaMA3-7B-Image",
    "VideoLLaMA3-2B-Image": "DAMO-NLP-SG/VideoLLaMA3-2B-Image"
}

# Start with image model (more stable)
selected_model = "VideoLLaMA3-2B-Image"
model_path = MODEL_OPTIONS[selected_model]

print(f"Loading model: {selected_model}")
print(f"Model path: {model_path}")

try:
    # Load model with optimized settings
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="flash_attention_2"
    ).to(device)
    
    # Try to load processor
    try:
        processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
        print("✅ Full processor loaded")
    except Exception as e:
        print(f"⚠️  Processor loading issue: {e}")
        print("💡 Falling back to tokenizer")
        processor = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        
    print(f"✅ Model loaded successfully on {device}!")
    print(f"Model parameters: {model.num_parameters():,}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n🔧 Troubleshooting:")
    print("1. Try: Runtime → Restart runtime")
    print("2. Try different model: VideoLLaMA3-2B-Image")
    print("3. Check GPU memory availability")
    print("4. If VideoInput error: use image-only models")
    raise e

## 4. Video Loading and Preprocessing Utilities

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from decord import VideoReader
import torch
from typing import List, Union

def load_video_frames(video_path: str, max_frames: int = 32, target_size: tuple = (224, 224)) -> List[Image.Image]:
    """Load video frames using decord for efficient video processing."""
    try:
        vr = VideoReader(video_path)
        total_frames = len(vr)
        
        # Sample frames evenly
        if total_frames <= max_frames:
            frame_indices = list(range(total_frames))
        else:
            step = total_frames / max_frames
            frame_indices = [int(i * step) for i in range(max_frames)]
        
        frames = []
        for idx in frame_indices:
            frame = vr[idx].asnumpy()
            frame = cv2.resize(frame, target_size)
            frame = Image.fromarray(frame)
            frames.append(frame)
            
        return frames
        
    except Exception as e:
        print(f"Error loading video {video_path}: {e}")
        return []

def download_sample_video() -> str:
    """Download a sample video for testing."""
    import gdown
    import os
    
    os.makedirs("sample_videos", exist_ok=True)
    video_url = "https://sample-videos.com/zip/10/mp4/360/SampleVideo_360x240_1mb.mp4"
    output_path = "sample_videos/sample_video.mp4"
    
    if not os.path.exists(output_path):
        print("Downloading sample video...")
        gdown.download(video_url, output_path, quiet=False)
        print(f"✅ Sample video downloaded to: {output_path}")
    else:
        print(f"✅ Sample video already exists: {output_path}")
        
    return output_path

def display_video_frames(frames: List[Image.Image], max_display: int = 8):
    """Display video frames in a grid."""
    num_frames = min(len(frames), max_display)
    cols = 4
    rows = (num_frames + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
        
    for i in range(num_frames):
        row, col = i // cols, i % cols
        axes[row, col].imshow(frames[i])
        axes[row, col].set_title(f"Frame {i+1}")
        axes[row, col].axis('off')
    
    for i in range(num_frames, rows * cols):
        row, col = i // cols, i % cols
        axes[row, col].axis('off')
        
    plt.tight_layout()
    plt.show()

print("✅ Video processing utilities loaded!")

## 5. Inference Function with Error Handling

In [ ]:
def run_inference(model, processor, video_path: str = None, images: List[Image.Image] = None, 
                 question: str = "What is happening in this video?", max_tokens: int = 200):
    """Run inference with comprehensive error handling."""
    try:
        # Prepare conversation format
        conversation = [
            {
                "role": "system",
                "content": "You are a helpful assistant for analyzing videos and images."
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": question}]
            }
        ]
        
        # Add media to conversation
        if video_path and not selected_model.endswith("Image"):
            frames = load_video_frames(video_path, max_frames=16)
            if frames:
                conversation[1]["content"].insert(0, {"type": "video", "video": frames})
                print(f"✅ Loaded {len(frames)} frames from video")
            else:
                print("❌ Failed to load video frames")
                return None
                
        elif images and selected_model.endswith("Image"):
            conversation[1]["content"].insert(0, {"type": "image", "image": images[0]})
            print(f"✅ Loaded {len(images)} image(s)")
        else:
            print("❌ Please provide either video_path or images")
            return None
            
        # Process inputs with fallback handling
        try:
            if hasattr(processor, '__call__') and 'image_processing' in str(type(processor)):
                inputs = processor(conversation=conversation, return_tensors="pt")
            else:
                print("⚠️  Using text-only mode")
                inputs = processor(question, return_tensors="pt")
                
        except Exception as e:
            print(f"⚠️  Processor failed: {e}")
            inputs = processor(question, return_tensors="pt")
            
        inputs = inputs.to(device)
        
        # Generate response
        print(f"🤖 Generating response to: '{question}'")
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                use_cache=True
            )
        
        # Decode response with fallback
        try:
            if hasattr(processor, 'batch_decode'):
                response = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
            else:
                response = processor.decode(output_ids[0], skip_special_tokens=True)
        except:
            response = "Response generated but decoding failed"
        
        return response.strip()
        
    except Exception as e:
        print(f"❌ Error during inference: {e}")
        print("🔧 Solutions: 1) Restart runtime 2) Try image model 3) Check GPU memory")
        return None

print("✅ Inference function ready!")

## 6. Test Examples

### 6.1 Image Understanding Test (Most Stable)

In [ ]:
# Test with image model (most stable option)
selected_model = "VideoLLaMA3-2B-Image"
model_path = MODEL_OPTIONS[selected_model]

print(f"Testing {selected_model} with image understanding...")

# Create a sample image
def create_sample_image():
    """Create a simple sample image for testing"""
    from PIL import Image, ImageDraw
    
    img = Image.new('RGB', (400, 300), color='lightblue')
    draw = ImageDraw.Draw(img)
    
    # Add some shapes and text
    draw.rectangle([50, 50, 200, 150], fill='red', outline='black')
    draw.ellipse([250, 50, 350, 150], fill='green', outline='black')
    draw.text((150, 200), "Hello VideoLLaMA3!", fill='black', anchor='mm')
    
    return img

sample_image = create_sample_image()
sample_image.save("sample_videos/test_image.png")
print("✅ Created sample test image")

# Display the image
plt.figure(figsize=(8, 6))
plt.imshow(sample_image)
plt.title("Sample Test Image")
plt.axis('off')
plt.show()

# Test image understanding
image_questions = [
    "What do you see in this image?",
    "Describe the shapes and colors present.",
    "What text can you read in this image?"
]

for i, question in enumerate(image_questions, 1):
    print(f"\n--- Image Test {i}: {question} ---")
    response = run_inference(model, processor, images=[sample_image], question=question)
    if response:
        print(f"🤖 Response: {response}")
    print("-" * 50)

### 6.2 Video Understanding Test (Advanced)

In [ ]:
# Try video model (may have import issues)
selected_model = "VideoLLaMA3-2B"
model_path = MODEL_OPTIONS[selected_model]

print(f"Testing {selected_model} with video understanding...")
print("Note: If this fails due to VideoInput import, stick with image models")

# Download sample video
video_path = download_sample_video()

# Test different types of questions
test_questions = [
    "What is happening in this video?",
    "Describe the main actions and events you see.",
    "What objects or people can you identify?"
]

# Load video frames for preview
sample_frames = load_video_frames(video_path, max_frames=8)
if sample_frames:
    print(f"📹 Video loaded with {len(sample_frames)} frames for preview")
    display_video_frames(sample_frames)

# Run inference tests
for i, question in enumerate(test_questions, 1):
    print(f"\n--- Test {i}: {question} ---")
    response = run_inference(model, processor, video_path=video_path, question=question)
    if response:
        print(f"🤖 Response: {response}")
    print("-" * 50)

## 7. Conclusion and Troubleshooting Guide

### 🎉 VideoLLaMA3 Colab Test Complete!

This notebook has successfully demonstrated VideoLLaMA3 with comprehensive error handling.

### 🔧 Troubleshooting Common Issues:

#### Issue 1: VideoInput Import Error
```
ImportError: cannot import name 'VideoInput' from 'transformers.image_utils'
```

**Solutions:**
1. **Use Image-Only Models**: `VideoLLaMA3-2B-Image` or `VideoLLaMA3-7B-Image`
2. **Restart Runtime**: Runtime → Restart runtime, then run cells in order
3. **Alternative**: The notebook includes fallback mechanisms for text-only processing

#### Issue 2: GPU Memory Errors
**Solutions:**
1. Use smaller models (`VideoLLaMA3-2B` variants)
2. Reduce frame count in video processing
3. Enable `torch.bfloat16` (already configured)

#### Issue 3: Model Loading Errors
**Solutions:**
1. Check internet connection for Hugging Face downloads
2. Ensure sufficient disk space in Colab
3. Try different model variants

### 💡 Recommendations:
- **Start with**: `VideoLLaMA3-2B-Image` (most stable)
- **For video tasks**: `VideoLLaMA3-2B` (may need troubleshooting)
- **Best quality**: `VideoLLaMA3-7B-Image` or `VideoLLaMA3-7B`

### 🚀 Next Steps:
1. **Upload your own images/videos** for custom testing
2. **Experiment with different question types**
3. **Try batch processing** for multiple files
4. **Fine-tune prompts** for your specific use case

### 🔗 Resources:
- [VideoLLaMA3 GitHub](https://github.com/DAMO-NLP-SG/VideoLLaMA3)
- [Model Documentation](https://huggingface.co/DAMO-NLP-SG)
- [Colab GPU Guide](https://colab.research.google.com/)

Happy exploring with VideoLLaMA3! 🤖📹✨